# Experiment 4: BiLSTM with Word Embedding<br>
## Objective:<br>
To evaluate the performance of a Bidirectional LSTM model for sentiment classification using word embeddings. LSTM model by itself performed poorly. (about 50% accuracy) Swapped to BiLSTM expecting an improvement because it can capture context from both directions (past and future), which improves understanding of sentence meaning. <br>
<br>
## Setup: <br>
- Feature: Embedding
- Model: BiLSTM
- Dataset: Amazon Reviews

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
print(os.getcwd())

c:\Users\USER\Desktop\FYP\fyp-nlp-reviews\experiments\embedding


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
df = pd.read_csv("../../data/processed/cleaned.csv")

In [6]:
X = df["clean_text"].fillna("").astype(str)
y = df["sentiment"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
max_words = 20000
max_len = 300

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [9]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [10]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128),
    Bidirectional(LSTM(128)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [11]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

In [12]:
model.fit(X_train_pad, y_train, epochs=10, batch_size=64,
          validation_split=0.2, callbacks=[early_stop])

Epoch 1/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 440s 749ms/step - accuracy: 0.8087 - loss: 0.4198 - val_accuracy: 0.8577 - val_loss: 0.3394
Epoch 2/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 512s 883ms/step - accuracy: 0.8870 - loss: 0.2849 - val_accuracy: 0.8654 - val_loss: 0.3294
Epoch 3/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 504s 870ms/step - accuracy: 0.9151 - loss: 0.2238 - val_accuracy: 0.8670 - val_loss: 0.3174
Epoch 4/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 501s 863ms/step - accuracy: 0.9319 - loss: 0.1799 - val_accuracy: 0.8657 - val_loss: 0.3476
Epoch 5/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 492s 849ms/step - accuracy: 0.9460 - loss: 0.1488 - val_accuracy: 0.8645 - val_loss: 0.3873


In [13]:
y_pred = (model.predict(X_test_pad) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

363/363 ━━━━━━━━━━━━━━━━━━━━ 43s 118ms/step
Accuracy: 0.8646435037503233
              precision    recall  f1-score   support

         0.0       0.83      0.91      0.87      5800
         1.0       0.90      0.82      0.86      5799

    accuracy                           0.86     11599
   macro avg       0.87      0.86      0.86     11599
weighted avg       0.87      0.86      0.86     11599

